In [1]:
from dotenv import load_dotenv
import os
load_dotenv()

True

In [2]:
from openai import OpenAI
openai_client = OpenAI()

In [4]:
from ingest import load_faq_data, build_index
from rag_helper import play


ImportError: cannot import name 'play' from 'rag_helper' (/workspaces/llm-zoomcamp-2026-code/rag_helper.py)

In [ ]:


documents = load_faq_data()
index = build_index(documents)

openai_client = OpenAI()

assistant = RAGBase(
    index=index,
    llm_client=openai_client,
)

answer = assistant.rag("I just discovered the course. Can I join now?")
print(answer)

ImportError: cannot import name 'RAGBase' from 'rag_helper' (/workspaces/llm-zoomcamp-2026-code/rag_helper.py)

In [11]:
def llm(prompt):
    response = openai_client.responses.create(
        model='gpt-5.4-mini',
        input=prompt
    )
    return getattr(response, "output_text", str(response))

In [12]:
prompt = 'I just discovered the course, can I join now?'

In [13]:
llm(prompt)

'Yes—you can usually join a course after it has started, but it depends on the course’s enrollment policy and where it is in the schedule.\n\nIf you want, I can help you figure out the best next step. For example, you could ask:\n\n- “Is it too late to enroll?”\n- “How do I join if the course already started?”\n- “Can I catch up on missed material?”\n\nIf you’re reaching out to the course instructor or support team, a good message would be:\n\n> Hi, I just discovered this course and wanted to ask if it’s still possible to join. If so, could you please let me know the steps? Thank you.\n\nIf you tell me the course/platform, I can help you draft a more specific message.'

In [14]:
context = """
I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we're still accepting submissions.

Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

What is the video/zoom link to the stream for the "Office Hours" or live/workshop sessions?
The zoom link is only published to instructors/presenters/TAs. Students participate via YouTube Live and submit questions to Slido.

Cloud alternatives with GPU
Check the quota and reset cycle carefully. Potential options include Google Colab, Kaggle, Databricks.
"""

In [16]:
question = 'I just discovered the course, can I join now?'

In [17]:
prompt = f"""
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."

Question:
{question}

Context:
{context}
"""

In [18]:
llm(prompt)

'Yes, you can still join now. If you want to receive a certificate, make sure to submit your project while submissions are still open.'

In [19]:
import requests

docs_url = "https://datatalks.club/faq/json/courses.json"
response = requests.get(docs_url)
courses_raw = response.json()

In [20]:
documents = []
url_prefix = "https://datatalks.club/faq"

for course in courses_raw:
    course_url = f"""{url_prefix}{course["path"]}"""

    course_response = requests.get(course_url)
    course_response.raise_for_status()
    course_data = course_response.json()

    documents.extend(course_data)

len(documents)

1342

In [21]:
documents[1010]

{'id': '862909f457',
 'course': 'machine-learning-zoomcamp',
 'section': 'Module 3. Machine Learning for Classification',
 'question': 'What is the better option FeatureHasher or DictVectorizer?',
 'answer': "These methods both receive a dictionary as input. While the `DictVectorizer` will store a large vocabulary and take up more memory, `FeatureHasher` creates vectors with a predefined length. They are both used for handling categorical features.\n\n- If you have high cardinality in categorical features, it's better to use `FeatureHasher`.\n- If you want to preserve feature names in transformed data and have a small number of unique values, use `DictVectorizer`.\n\nYour choice will depend on your data. For more information, you can visit [scikit-learn.org](https://scikit-learn.org/stable/auto_examples/text/plot_hashing_vs_dict_vectorizer.html)"}

In [22]:
from minsearch import Index

index = Index(
    text_fields=['course', 'section', 'question', 'answer'],
    keyword_fields=['course']
)
index.fit(documents)

In [23]:
def search(question):
    boost_dict={'question': 2.0, 'section': 0.5}
    filter_dict={'course': 'llm-zoomcamp'}

    results = index.search(question,
                           num_results=5,
                           boost_dict=boost_dict,
                           filter_dict=filter_dict,
                           )
    return results    

In [24]:
search_results = search(question)
search_results

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '977bf7786c',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
  'answer': "You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date."},
 {'id': '69d122f12e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
  'answer': 'No, you c

In [25]:
INSTRUCTIONS = """
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."
"""

In [26]:
USER_PROMPT_TEMPLATE = '''
question:
{question}

context: 
{context}
'''

In [31]:
search_results

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '977bf7786c',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
  'answer': "You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date."},
 {'id': '69d122f12e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
  'answer': 'No, you c

In [ ]:
def build_context(search_results):
    lines = []
    for doc in search_results:
        lines.append(doc['section'])
        lines.append('Q: ' + doc['question'])
        lines.append('A: ' + doc['answer'])
        lines.append(' ')
    return '\n'.join(lines)

    

In [41]:
def build_prompt(question, search_results):
    context = build_context(search_results)
    prompt = USER_PROMPT_TEMPLATE.format(question=question, context=context)
    return prompt

In [42]:
prompt = build_prompt(question, search_results)
print(prompt)


question:
I just discovered the course, can I join now?

context: 
General Course-Related Questions
Q: I just discovered the course. Can I still join?
A: Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.
 
General Course-Related Questions
Q: Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
A: You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.
 
General Course-Related Questions
Q: Certificate: Can I follow the course in a self-paced mode and get a certificate?
A: No, you can only get a certificate if you finish the course with a "live" cohort.

We don't award certificates for the self-paced mode. The reason is you need to peer-review 3 capstone(s) after submitting your pro

In [43]:
response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=prompt
)

In [49]:
response.output_text

'Yes — you can still join now.\n\nIf you want a certificate, though, you need to submit your project while submissions are still open. If the course is no longer running live, you can still follow it, but you won’t be able to get a certificate in self-paced mode.'

In [50]:
response.usage

ResponseUsage(input_tokens=341, input_tokens_details=InputTokensDetails(cached_tokens=0), output_tokens=62, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=403)

In [56]:
print(response.model_dump_json(indent=2))

{
  "id": "resp_021ec93efe0c25b8006a2fdc5440008193848f16402fc68541",
  "created_at": 1781521492.0,
  "error": null,
  "incomplete_details": null,
  "instructions": null,
  "metadata": {},
  "model": "gpt-5.4-mini-2026-03-17",
  "object": "response",
  "output": [
    {
      "id": "msg_021ec93efe0c25b8006a2fdc54d9a881939e50fc88029cb04e",
      "content": [
        {
          "annotations": [],
          "text": "Yes — you can still join now.\n\nIf you want a certificate, though, you need to submit your project while submissions are still open. If the course is no longer running live, you can still follow it, but you won’t be able to get a certificate in self-paced mode.",
          "type": "output_text",
          "logprobs": []
        }
      ],
      "role": "assistant",
      "status": "completed",
      "type": "message",
      "phase": "final_answer"
    }
  ],
  "parallel_tool_calls": true,
  "temperature": 1.0,
  "tool_choice": "auto",
  "tools": [],
  "top_p": 0.98,
  "backgr

In [59]:
response.usage.input_tokens, response.usage.output_tokens, response.usage.total_tokens

(341, 62, 403)

In [60]:
input_price = 0.75 / 1_000_000
output_price = 4.50 / 1_000_000

cost = response.usage.input_tokens * input_price + response.usage.output_tokens * output_price
cost

0.0005347500000000001

In [75]:
def llm(INSTRUCTIONS, prompt, model="gpt-5.4-mini"):
    message_history = [
        {"role": "developer", "content": INSTRUCTIONS},
        {"role": "user", "content": prompt},
        ]
    
    response = openai_client.responses.create(
        model=model,
        input=message_history
        )   
    return getattr(response, "output_text", str(response))

In [76]:
def rag(question, instructions=INSTRUCTIONS, model="gpt-5.4-mini"):
    search_results = search(question)
    prompt = build_prompt(question, search_results)
    answer = llm(instructions, prompt, model=model)
    return answer

In [77]:
answer = rag(question)
answer

'Yes, but if you want to receive a certificate, you need to submit your project while submissions are still open.'

In [73]:
message_history = [
    {"role": "developer", "content": INSTRUCTIONS},
    {"role": "user", "content": prompt},
    ]

In [74]:
message_history

[{'role': 'developer',
  'content': '\nYour task is to answer questions from the course participants\nbased on the provided context.\n\nUse the context to find relevant information and provide accurate\nanswers. If the answer is not found in the context,\nrespond with "I don\'t know."\n'},
 {'role': 'user',
  'content': '\nquestion:\nI just discovered the course, can I join now?\n\ncontext: \nGeneral Course-Related Questions\nQ: I just discovered the course. Can I still join?\nA: Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.\n \nGeneral Course-Related Questions\nQ: Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?\nA: You don\'t need it. You\'re accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.